# Manufacturing Analytics — Exploratory Analysis

This notebook demonstrates the platform's Python API directly (as an alternative to running `main.py`), and explores the generated dataset interactively.

Run the full pipeline first (`python main.py`) or execute the cells below to generate/load data from scratch.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
pd.set_option('display.max_columns', 50)

from manufacturing_analytics.utils.config import load_config
from manufacturing_analytics.data_generation.generator import ManufacturingDataGenerator
from manufacturing_analytics.data_processing.cleaning import DataCleaner
from manufacturing_analytics.kpi.calculations import KPICalculator
from manufacturing_analytics.visualization.charts import ChartFactory

config = load_config('../config/config.yaml')
config['data_generation'].keys()

## 1. Generate (or load) the dataset

In [ ]:
import os

raw_dir = '../data/raw'
if os.path.exists(os.path.join(raw_dir, 'production_records.csv')):
    production = pd.read_csv(os.path.join(raw_dir, 'production_records.csv'))
    downtime = pd.read_csv(os.path.join(raw_dir, 'downtime_events.csv'))
    inspections = pd.read_csv(os.path.join(raw_dir, 'quality_inspections.csv'))
    defects = pd.read_csv(os.path.join(raw_dir, 'defects.csv'))
    machines = pd.read_csv(os.path.join(raw_dir, 'machines.csv'))
    print('Loaded existing generated dataset from data/raw/')
else:
    generator = ManufacturingDataGenerator(config)
    dataset = generator.generate_all()
    production = dataset.production_records
    downtime = dataset.downtime_events
    inspections = dataset.quality_inspections
    defects = dataset.defects
    machines = dataset.machines
    print('Generated a fresh dataset in-memory.')

production.shape, downtime.shape, inspections.shape, defects.shape

In [ ]:
production.head()

## 2. Clean the data

In [ ]:
cleaner = DataCleaner(config)

clean_production, report = cleaner.clean(
    production, table_name='production',
    date_cols=['date'],
    numeric_cols=['planned_production_time_minutes', 'downtime_minutes', 'run_time_minutes',
                  'total_units_produced', 'good_units', 'scrap_units'],
    outlier_cols=['downtime_minutes', 'total_units_produced'],
    non_negative_cols=['total_units_produced', 'good_units', 'scrap_units'],
)
print(report.summary())
clean_production.head()

## 3. Compute KPIs

In [ ]:
kpi_calc = KPICalculator(config)
with_oee = kpi_calc.add_oee_components(clean_production)
with_oee['date'] = pd.to_datetime(with_oee['date'])
with_oee['month'] = with_oee['date'].dt.to_period('M').astype(str)

monthly_kpis = kpi_calc.aggregate_kpis(with_oee, group_by=['month'])
factory_kpis = kpi_calc.aggregate_kpis(with_oee, group_by=['factory_id'])
monthly_kpis[['month', 'oee', 'availability', 'performance', 'quality']]

In [ ]:
mtbf_mttr = kpi_calc.calculate_mtbf_mttr(downtime, with_oee, group_by=['machine_id'])
mtbf_mttr.sort_values('mtbf_hours').head(10)

## 4. Visualize

In [ ]:
cf = ChartFactory()
fig = cf.line_chart(monthly_kpis.sort_values('month'), x='month', y='oee', title='Monthly OEE Trend')
fig.show()

In [ ]:
pareto = kpi_calc.pareto_analysis(defects, category_col='defect_type')
fig = cf.pareto_chart(pareto, category_col='defect_type', title='Defect Pareto Analysis')
fig.show()

In [ ]:
fig = cf.box_chart(with_oee, x='factory_id', y='oee', title='OEE Distribution by Factory')
fig.show()

## 5. Control chart (SPC) example

Useful for spotting when a process has drifted outside statistical control limits (+/-3 sigma).

In [ ]:
fig = cf.control_chart(monthly_kpis.sort_values('month'), x='month', y='oee', title='OEE Control Chart')
fig.show()

## Next steps

- Run `python ../main.py` from the project root for the full automated pipeline (dashboard + Excel + PDF reports).
- See `docs/architecture.md` for how this notebook's building blocks map onto the production pipeline in `main.py`.